In [86]:
import os
import sys 
import torch
from pathlib import Path

from satchange import (
    BCEDiceloss
)

from satchange.models import FCSiamDiff

from torch.utils.data import DataLoader
from satchange.utils import load_config
from satchange.data import (
    RandomFlipRotate,
    prepare_levir_cd, 
    LEVIRCDDataset,
)
from satchange.experiments import (
    load_fc_siam_diff_experiment
)

In [87]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [88]:
print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Selected GPU:", torch.cuda.get_device_name(0))

Python executable: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/.venv/bin/python
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
PyTorch: 2.12.1+cu130
PyTorch CUDA: 13.0
CUDA available: True
Visible GPU count: 1
Selected GPU: NVIDIA A100-SXM4-80GB


In [89]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

full_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_full.yaml"
)

resized_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_resized.yaml"
)


print("The project root: ", project_root)

print("Full config:", full_config_path)
print("Full config exists:", full_config_path.exists())

print("Resized config:", resized_config_path)
print("Resized config exists:", resized_config_path.exists())

The project root:  /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence
Full config: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/fc_siam_diff_full.yaml
Full config exists: True
Resized config: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/fc_siam_diff_resized.yaml
Resized config exists: True


### Resized Dataset

In [90]:
EXPERIMENT = "resized"
# Available "full" and "resized"


config, config_path = load_fc_siam_diff_experiment(
    experiment=EXPERIMENT,
    project_root=project_root,
)

print("Experiment:", EXPERIMENT)
print("Configuration:", config_path)
print("Image size:", config["data"]["image_size"])
print("Batch size:", config["data"]["batch_size"])
print("Output directory:", config["output"]["directory"])

Experiment: resized
Configuration: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/fc_siam_diff_resized.yaml
Image size: 256
Batch size: 2
Output directory: artifacts/fc_siam_diff_resized


In [91]:
# path to the dataset
dataset_root = prepare_levir_cd(
    config=config,
    project_root=project_root,
)

print("Ready dataset root:", dataset_root)

Download directory is not empty: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd
Skipping download.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/train
Skipping extraction.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/val
Skipping extraction.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/test
Skipping extraction.

Split:         train
Before images: 445
After images:  445
Labels:        445

Split:         val
Before images: 64
After images:  64
Labels:        64

Split:         test
Before images: 128
After images:  128
Labels:        128

Total samples: 637
LEVIR-CD structure is valid.
Ready dataset root: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD


In [92]:
train_transform = RandomFlipRotate(
    horizontal_flip_probability = 0.5,
    vertical_flip_probability = 0.5,
)

train_dataset = LEVIRCDDataset(
    dataset_root=dataset_root, 
    split="train",
    image_size=config["data"]["image_size"],
    transform=train_transform
)

In [93]:
validation_dataset = LEVIRCDDataset(
    dataset_root=dataset_root,
    split="val",
    image_size=config["data"]["image_size"],
    transform=None,
)

test_dataset = LEVIRCDDataset(
    dataset_root=dataset_root,
    split="test",
    image_size=config["data"]["image_size"],
    transform=None,
)

In [94]:
num_workers = config["data"]["num_workers"]
batch_size = config["data"]["batch_size"]

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

In [95]:
train_batch = next(iter(train_loader))

print("Before:", train_batch["before"].shape)
print("After:", train_batch["after"].shape)
print("Mask:", train_batch["mask"].shape)
print("Sample IDs:", train_batch["sample_id"])

Before: torch.Size([2, 3, 256, 256])
After: torch.Size([2, 3, 256, 256])
Mask: torch.Size([2, 1, 256, 256])
Sample IDs: ['train_169', 'train_57']


In [96]:
before = train_batch["before"].to(
    device,
    non_blocking=True,
)

after = train_batch["after"].to(
    device,
    non_blocking=True,
)

mask = train_batch["before"].to(
    device,
    non_blocking=True,
)

In [97]:
model = FCSiamDiff(
    in_channels=config["model"]["parameters"]["in_channels"],
    out_channels=config["model"]["parameters"]["out_channels"],
    base_channels=config["model"]["parameters"]["base_channels"],
).to(device)

loss_function = BCEDiceloss(
    bce_weight=config["training"]["loss"]["bce_weight"],
    dice_weight=config["training"]["loss"]["dice_weight"],
).to(device)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"]
)

In [98]:
def train_one_epoch(
    model: torch.nn.Module,
    data_loader: DataLoader,
    loss_function: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    """Train the model for one complete epoch."""
    
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in data_loader:
        before = batch["before"].to(
            device,
            non_blocking=True,
        )
        after = batch["after"].to(
            device,
            non_blocking=True,
        )
        mask = batch["mask"].to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True,
        )

        logits = model(
            before,
            after,
        )

        loss = loss_function(
            logits,
            mask,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Training loss is not finite: {loss.item()}"
            )

        loss.backward()
        optimizer.step()

        current_batch_size = before.shape[0]

        total_loss += (
            loss.item() * current_batch_size
        )
        total_samples += current_batch_size

        return total_loss / total_samples

In [99]:
def validate_one_epoch(
    model: torch.nn.Module,
    data_loader: DataLoader,
    loss_function: torch.nn.Module,
    device: torch.device,
) -> float:
    """Evaluate the model for one complete validation epoch."""

    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in data_loader:
            before = batch["before"].to(
            device,
            non_blocking=True,
            )
            after = batch["after"].to(
                device,
                non_blocking=True,
            )
            mask = batch["mask"].to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )

            logits = model(
                before,
                after,
            )

            loss = loss_function(
                logits, 
                mask,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Validation loss is not finite: {loss.item()}"
                )

            current_batch_size = before.shape[0]

            total_loss += (
                loss.item() * current_batch_size
            )
            total_samples += current_batch_size
        
        return total_loss / total_samples
        

In [100]:
train_loss = train_one_epoch(
    model=model,
    data_loader=train_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    device=device,
)

validation_loss = validate_one_epoch(
    model=model,
    data_loader=validation_loader,
    loss_function=loss_function,
    device=device,
)


print(f"Train loss: {train_loss:.6f}")
print(f"Validation loss: {validation_loss:.6f}")

Train loss: 0.816016
Validation loss: 0.785344


In [101]:
# before_features = model._encode(before)
# after_features = model._encode(after)

# feature_differences = model._compute_feature_differences(
#     before_features,
#     after_features,
# )

# decoded_features = model._decode(
#     feature_differences
# )

# print(decoded_features.shape)

In [102]:
# # switches the model into training mode
# model.train()
# # and
# model.eval()

# for batch in train_loader:
#     # cleares gradients left from previous batches 'cause PyTorch accumulates gradients
#     optimizer.zero_grad()

#     # the prediction and computation graph
#     prediction = model(
#         batch["before"],
#         batch["after"],
#     )

#     # computing the loss (how different the prediction from the real mask)
#     loss = loss_function(
#         prediction,
#         batch["mask"],
#     )

#     # computing the gradient (what change to reduce the loss)
#     loss.backward()

#     # update the weights using gradients
#     optimizer.step()


# # PyTorch cycle that does not effect the model. It's used to evaluate but not to train.
# with torch.no_grad():
#     for batch in validation_loader:
#         prediction = model(
#             batch["before"],
#             batch["after"],
#         )

#         # computing the loss
#         loss = loss_function(
#             prediction,
#             batch["mask"]
#         )

### Full sized Dataset

In [103]:
# EXPERIMENT = "full"

# config, config_path = load_fc_siam_diff_experiment(
#     experiment=EXPERIMENT,
#     project_root=project_root,
# )

# print(f"Configuration {config_path}")
# print(f"Data settings: {config['data']}")

In [104]:
# dataset_root = prepare_levir_cd(
#     config=config,
#     project_root=project_root,
# )


# dataset = LEVIRCDDataset(
#     dataset_root=dataset_root,
#     split="train",
#     image_size=config["data"]["image_size"],
# )

# sample=dataset[0]


# print(f"Number of samples: {len(dataset)}")
# print(f"Sample ID: {sample['sample_id']}")
# print(f"Before shape: {sample['before'].shape}")
# print(f"After shape: {sample['after'].shape}")
# print(f"Mask shape: {sample['mask'].shape}")

In [105]:
# before_features = model._encode(before)
# after_features = model._encode(after)

# feature_differences = model._compute_feature_differences(
#     before_features,
#     after_features,
# )

# decoded_features = model._decode(
#     feature_differences
# )

# print(decoded_features.shape)

In [106]:
# # switches the model into training mode
# model.train()
# # and
# model.eval()

# for batch in train_loader:
#     # cleares gradients left from previous batches 'cause PyTorch accumulates gradients
#     optimizer.zero_grad()

#     # the prediction and computation graph
#     prediction = model(
#         batch["before"],
#         batch["after"],
#     )

#     # computing the loss (how different the prediction from the real mask)
#     loss = loss_function(
#         prediction,
#         batch["mask"],
#     )

#     # computing the gradient (what change to reduce the loss)
#     loss.backward()

#     # update the weights using gradients
#     optimizer.step()


# # PyTorch cycle that does not effect the model. It's used to evaluate but not to train.
# with torch.no_grad():
#     for batch in validation_loader:
#         prediction = model(
#             batch["before"],
#             batch["after"],
#         )

#         # computing the loss
#         loss = loss_function(
#             prediction,
#             batch["mask"]
#         )

### Full sized Dataset

In [107]:
# EXPERIMENT = "full"

# config, config_path = load_fc_siam_diff_experiment(
#     experiment=EXPERIMENT,
#     project_root=project_root,
# )

# print(f"Configuration {config_path}")
# print(f"Data settings: {config['data']}")

In [108]:
# dataset_root = prepare_levir_cd(
#     config=config,
#     project_root=project_root,
# )


# dataset = LEVIRCDDataset(
#     dataset_root=dataset_root,
#     split="train",
#     image_size=config["data"]["image_size"],
# )

# sample=dataset[0]


# print(f"Number of samples: {len(dataset)}")
# print(f"Sample ID: {sample['sample_id']}")
# print(f"Before shape: {sample['before'].shape}")
# print(f"After shape: {sample['after'].shape}")
# print(f"Mask shape: {sample['mask'].shape}")